### Jav-Nairobi (Temporal Equity)

### i. libraries

In [1]:
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt
import seaborn as sns
import gtfs_kit as gk

### ii. background

Temporal equity looks at how service availability varies over time, especially during peak vs. off-peak hours, and across different regions or population groups.

The main questions:

- Do all population groups have similar access to transit throughout the day or week?
- Do low-income or peripheral areas get fewer trips per hour?
- Are night or weekend services equally distributed?
- Do peak-hour frequencies differ by income zone?

### iii. data

In [2]:
# this dataframe contains the full data from the coverage eda
ward_full_gdf = pd.read_csv('/home/dataopske/Desktop/jav/data/processed/wards_full_gdf.csv')

# load GTFS data
feed_path = '/home/dataopske/Desktop/jav/data/raw/digitalmatatu/GTFS_FEED_2019.zip'
feed = gk.read_feed(feed_path,dist_units='km')

### iv. Descriptive

#### a. Trips per hour per route

In [3]:
frequencies=feed.frequencies

In [4]:
frequencies.head(1)

,trip_id,start_time,end_time,headway_secs
0,10106110,06:00:00,09:00:00,300


- `Headway` is the time gap between two consecutive vehicles (buses, matatus, trains, etc.) on the same route.
- `Hour` simply means which hour of the day that trip starts — derived from start_time in the GTFS data

In [5]:
# Clean start_time to hour
frequencies['start_time'] = pd.to_timedelta(frequencies['start_time'], errors='coerce')
frequencies['hour'] = frequencies['start_time'].dt.seconds // 3600

# Average headway per route-hour 
freq_hour = (
    frequencies.groupby(['trip_id', 'hour'])
    ['headway_secs'].mean()
    .reset_index()
)

# Convert headway to trips/hour
freq_hour['trips_per_hour'] = 3600 / freq_hour['headway_secs']


`trips_per_hour` is the number of vehicles leaving per hour on a route — a direct measure of how frequent the service is.

In [6]:
freq_hour.head(2)

,trip_id,hour,headway_secs,trips_per_hour
0,10106110,6,300.0,12.0
1,10106110,9,900.0,4.0


So if a ward has an average `trips_per_hour` = 4 at 8 AM, that means on average, `four vehicles` serve that ward per hour; 
that’s decent coverage during that time.

#### b. Trips per ward

In [7]:
stop_times=feed.stop_times

In [8]:
stops_df = feed.stops
stops_df.head(2)

,stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station
0,0001RLW,Railways,-1.290884,36.828242,0,<NA>
1,0002KOJ,Koja,-1.281230,36.822596,1,<NA>


In [10]:
from shapely import wkt
import geopandas as gpd

# Step 1: Connect trips to stops
trip_stops = stop_times[['trip_id', 'stop_id']].drop_duplicates()

# Step 2: Add frequency data to each stop
stop_service = trip_stops.merge(
    freq_hour[['trip_id', 'hour', 'trips_per_hour']], 
    on='trip_id',
    how='inner'
)

# Step 3: Aggregate by stop and hour
stop_hour_service = (
    stop_service
    .groupby(['stop_id', 'hour'], as_index=False)
    ['trips_per_hour'].sum()
)

# Step 4: Convert ward geometries from strings to Shapely objects
ward_full_gdf['geometry'] = ward_full_gdf['geometry'].apply(wkt.loads)

# Step 5: Set correct CRS for wards (UTM 37S for Nairobi)
ward_full_gdf = gpd.GeoDataFrame(
    ward_full_gdf,
    geometry='geometry',
    crs="EPSG:32737"
)

# Step 6: Create stops in WGS84 and transform to match wards
stops_gdf = gpd.GeoDataFrame(
    stops_df,
    geometry=gpd.points_from_xy(stops_df.stop_lon, stops_df.stop_lat),
    crs="EPSG:4326"
)
stops_gdf = stops_gdf.to_crs(ward_full_gdf.crs)

# Step 7: Spatial join - assign stops to wards
stops_with_ward = gpd.sjoin(
    stops_gdf,
    ward_full_gdf[['ward', 'population', 'geometry']],
    how='inner',
    predicate='within'
)

# Step 8: Add service frequency to stops
stops_with_service = stops_with_ward.merge(
    stop_hour_service,
    on='stop_id',
    how='inner'
)

# Step 9: Sum service by ward and hour
service_by_ward_hour = (
    stops_with_service
    .groupby(['ward', 'hour'], as_index=False)
    .agg({
        'trips_per_hour': 'sum',
        'population': 'first'
    })
)

# Step 10: Calculate service per capita
service_by_ward_hour['service_per_1k_pop'] = (
    service_by_ward_hour['trips_per_hour'] / 
    service_by_ward_hour['population'] * 1000
)

print(f"Final result: {len(service_by_ward_hour)} ward-hour combinations")
print(service_by_ward_hour.head(10))

Final result: 237 ward-hour combinations
              ward  hour  trips_per_hour     population  service_per_1k_pop
0     Airbase Ward     6          1164.0  105433.255157            11.04016
1     Airbase Ward     9           388.0  105433.255157            3.680053
2     Airbase Ward    15          1164.0  105433.255157            11.04016
3        Babandogo     6           504.0  144280.941622            3.493185
4        Babandogo     9           168.0  144280.941622            1.164395
5        Babandogo    15           504.0  144280.941622            3.493185
6  California Ward     6            36.0   46558.616489            0.773219
7  California Ward     9            12.0   46558.616489             0.25774
8  California Ward    15            36.0   46558.616489            0.773219
9        Clay City     6           732.0   59710.233673           12.259205


In [32]:
# Average service per ward (across all hours)
avg_service_by_ward = service_by_ward_hour.groupby('ward').agg({
    'service_per_1k_pop': 'mean',
    'trips_per_hour': 'sum',
    'population': 'first'
}).reset_index()

In [33]:
avg_service_by_ward.shape

(79, 4)

In [34]:
service_by_ward_hour

,ward,hour,trips_per_hour,population,service_per_1k_pop,trips_per_person_per_hour,trips_per_1k_people_per_hour
0,Airbase Ward,6,1164.0,105433.255157,11.04016,0.01104,11.04016
1,Airbase Ward,9,388.0,105433.255157,3.680053,0.00368,3.680053
2,Airbase Ward,15,1164.0,105433.255157,11.04016,0.01104,11.04016
3,Babandogo,6,504.0,144280.941622,3.493185,0.003493,3.493185
4,Babandogo,9,168.0,144280.941622,1.164395,0.001164,1.164395
...,...,...,...,...,...,...,...
232,Zimmerman Ward,9,60.0,41029.548004,1.462361,0.001462,1.462361
233,Zimmerman Ward,15,180.0,41029.548004,4.387082,0.004387,4.387082
234,Ziwani/kariokor,6,300.0,0.000000,inf,inf,inf
235,Ziwani/kariokor,9,100.0,0.000000,inf,inf,inf


In [35]:
# Per-capita trips per person per hour
service_by_ward_hour['trips_per_person_per_hour'] = (
    service_by_ward_hour['trips_per_hour'] / service_by_ward_hour['population']
)


In [36]:

service_by_ward_hour

,ward,hour,trips_per_hour,population,service_per_1k_pop,trips_per_person_per_hour,trips_per_1k_people_per_hour
0,Airbase Ward,6,1164.0,105433.255157,11.04016,0.01104,11.04016
1,Airbase Ward,9,388.0,105433.255157,3.680053,0.00368,3.680053
2,Airbase Ward,15,1164.0,105433.255157,11.04016,0.01104,11.04016
3,Babandogo,6,504.0,144280.941622,3.493185,0.003493,3.493185
4,Babandogo,9,168.0,144280.941622,1.164395,0.001164,1.164395
...,...,...,...,...,...,...,...
232,Zimmerman Ward,9,60.0,41029.548004,1.462361,0.001462,1.462361
233,Zimmerman Ward,15,180.0,41029.548004,4.387082,0.004387,4.387082
234,Ziwani/kariokor,6,300.0,0.000000,inf,inf,inf
235,Ziwani/kariokor,9,100.0,0.000000,inf,inf,inf


In [30]:
service_by_ward_hour['trips_per_1k_people_per_hour'] = (
    service_by_ward_hour['trips_per_person_per_hour'] * 1000
)

In [43]:
import numpy as np

def gini_coefficient(x):
    """Compute Gini coefficient safely (avoids divide-by-zero)."""
    x = np.array(x, dtype=float)
    x = x[~np.isnan(x)]  # drop NaNs
    if len(x) == 0:
        return np.nan
    total = np.sum(x)
    if total == 0:
        return 0.0  # no service = perfectly equal (everyone gets 0)
    x = np.sort(x)
    n = len(x)
    cumx = np.cumsum(x)
    gini = (n + 1 - 2 * np.sum(cumx) / cumx[-1]) / n
    return gini


In [44]:
temporal_equity = (
    service_by_ward_hour
    .groupby('hour')['trips_per_person_per_hour']
    .apply(gini_coefficient)
    .reset_index(name='gini')
)

/tmp/ipykernel_10304/852376135.py:15: RuntimeWarning: invalid value encountered in scalar divide
  gini = (n + 1 - 2 * np.sum(cumx) / cumx[-1]) / n
/tmp/ipykernel_10304/852376135.py:15: RuntimeWarning: invalid value encountered in scalar divide
  gini = (n + 1 - 2 * np.sum(cumx) / cumx[-1]) / n
/tmp/ipykernel_10304/852376135.py:15: RuntimeWarning: invalid value encountered in scalar divide
  gini = (n + 1 - 2 * np.sum(cumx) / cumx[-1]) / n


In [41]:
def weighted_gini(x, w):
    """Weighted Gini coefficient."""
    sorted_idx = np.argsort(x)
    x, w = np.array(x)[sorted_idx], np.array(w)[sorted_idx]
    cumw = np.cumsum(w)
    cumxw = np.cumsum(x * w)
    return 1 - 2 * np.sum(cumxw * w) / (cumxw[-1] * cumw[-1]) + (np.sum(w ** 2) / cumw[-1] ** 2)


In [42]:
temporal_equity_weighted = (
    service_by_ward_hour
    .groupby('hour')
    .apply(lambda g: weighted_gini(g['trips_per_person_per_hour'], g['population']))
    .reset_index(name='weighted_gini')
)


/tmp/ipykernel_10304/4086747181.py:6: RuntimeWarning: invalid value encountered in multiply
  cumxw = np.cumsum(x * w)
/tmp/ipykernel_10304/4086747181.py:6: RuntimeWarning: invalid value encountered in multiply
  cumxw = np.cumsum(x * w)
/tmp/ipykernel_10304/4086747181.py:6: RuntimeWarning: invalid value encountered in multiply
  cumxw = np.cumsum(x * w)
/tmp/ipykernel_10304/336909818.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: weighted_gini(g['trips_per_person_per_hour'], g['population']))


In [ ]:
(service_by_ward_hour.groupby('hour')['trips_per_person_per_hour']
 .sum()
 .reset_index(name='total_service')
 .query('total_service == 0'))

,hour,total_service
